In [5]:
import cv2
import numpy as np
import os

os.makedirs("output", exist_ok=True)

cap = cv2.VideoCapture("v5.mp4")

fourcc = cv2.VideoWriter_fourcc(*'XVID')

out = cv2.VideoWriter(
    "output/final.avi",
    fourcc,
    20.0,
    (640, 480)
)

def enhance(frame):

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    blur = cv2.GaussianBlur(
        gray,
        (5,5),
        0
    )

    clahe = cv2.createCLAHE(
        clipLimit=3.0,
        tileGridSize=(8,8)
    ).apply(blur)

    return gray, clahe

def road_mask(frame):

    h, w = frame.shape[:2]

    mask = np.zeros((h,w), dtype=np.uint8)

    mask[
        int(h*0.45):h,
        int(w*0.08):int(w*0.92)
    ] = 255

    return mask

def detect_cracks(gray, mask):

    roi = cv2.bitwise_and(
        gray,
        gray,
        mask=mask
    )

    blur = cv2.GaussianBlur(
        roi,
        (7,7),
        0
    )

    edges = cv2.Canny(
        blur,
        45,
        130
    )

    return edges

def detect_edge_cracks(gray, output, mask):

    cracks = detect_cracks(gray, mask)

    h, w = gray.shape

    roi = cracks[int(h*0.45):h, :]

    edge_roi = np.zeros_like(roi)

    edge_roi[:, :int(w*0.20)] = roi[:, :int(w*0.20)]

    edge_roi[:, int(w*0.80):] = roi[:, int(w*0.80):]

    crack_map = np.zeros_like(edge_roi)

    block_w = 40
    block_h = 80

    for y in range(0, edge_roi.shape[0]-block_h, 20):

        for x in range(0, edge_roi.shape[1]-block_w, 20):

            region = edge_roi[
                y:y+block_h,
                x:x+block_w
            ]

            density = np.mean(region > 0)

            if density > 0.07:

                crack_map[
                    y:y+block_h,
                    x:x+block_w
                ] = 255

    crack_map = cv2.morphologyEx(
        crack_map,
        cv2.MORPH_CLOSE,
        np.ones((25,25), np.uint8),
        iterations=2
    )

    contours, _ = cv2.findContours(
        crack_map,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    count = 0

    for c in contours:

        area = cv2.contourArea(c)

        if area < 2500:
            continue

        x, y, wc, hc = cv2.boundingRect(c)

        y = y + int(h*0.45)

        aspect = hc / (wc + 1e-5)

        if aspect < 1.2:
            continue

        cv2.rectangle(
            output,
            (x,y),
            (x+wc,y+hc),
            (255,255,0),
            3
        )

        cv2.putText(
            output,
            "Edge Crack",
            (x,y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255,255,0),
            2
        )

        count += 1

    return count

def detect_alligator(gray, output, mask):

    cracks = detect_cracks(gray, mask)

    h, w = gray.shape

    roi = cracks[int(h*0.40):h, :]

    block = 50

    crack_map = np.zeros_like(roi)

    for y in range(0, roi.shape[0]-block, 25):

        for x in range(int(w*0.20), int(w*0.80)-block, 25):

            region = roi[y:y+block, x:x+block]

            d = np.mean(region > 0)

            if d > 0.15:

                crack_map[
                    y:y+block,
                    x:x+block
                ] = 255

    crack_map = cv2.morphologyEx(
        crack_map,
        cv2.MORPH_CLOSE,
        np.ones((21,21), np.uint8),
        2
    )

    contours, _ = cv2.findContours(
        crack_map,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    count = 0

    for c in contours:

        if cv2.contourArea(c) < 4000:
            continue

        x,y,wc,hc = cv2.boundingRect(c)

        y = y + int(h*0.40)

        aspect = wc / (hc + 1e-5)

        if aspect < 0.7:
            continue

        cv2.rectangle(
            output,
            (x,y),
            (x+wc,y+hc),
            (0,255,0),
            2
        )

        cv2.putText(
            output,
            "Alligator",
            (x,y-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,255,0),
            2
        )

        count += 1

    return count


def detect_pothole(gray, output, mask):

    blur = cv2.GaussianBlur(gray, (7,7), 0)

    _, th = cv2.threshold(
        blur,
        80,
        255,
        cv2.THRESH_BINARY_INV
    )

    th = cv2.bitwise_and(th, th, mask=mask)

    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((7,7), np.uint8))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((17,17), np.uint8))

    h_profile = np.sum(th, axis=1)

    if np.std(h_profile) > 0:
        norm = (h_profile - np.mean(h_profile)) / (np.std(h_profile) + 1e-5)
    else:
        norm = h_profile

    stripe_score = np.sum(np.abs(np.diff(norm)) > 1.2)

    if stripe_score > 18:
        return 0

    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    count = 0

    for c in contours:

        area = cv2.contourArea(c)

        if area < 1800:
            continue

        x, y, w, h = cv2.boundingRect(c)

        ratio = w / (h + 1e-5)

        if ratio > 3.0:
            continue

        roi = gray[y:y+h, x:x+w]
        if np.mean(roi) > 235:
            continue

        cv2.rectangle(output, (x,y), (x+w,y+h), (0,0,255), 2)

        cv2.putText(
            output,
            "Pothole",
            (x,y-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,0,255),
            2
        )

        count += 1

    return count

def detect_faded_crossing(frame, output, mask):

    roi = cv2.bitwise_and(
        frame,
        frame,
        mask=mask
    )

    gray = cv2.cvtColor(
        roi,
        cv2.COLOR_BGR2GRAY
    )

    blur = cv2.GaussianBlur(
        gray,
        (5,5),
        0
    )

    edges = cv2.Canny(
        blur,
        50,
        140
    )

    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi/180,
        threshold=100,
        minLineLength=120,
        maxLineGap=20
    )

    if lines is None:
        return 0

    y_vals = []
    x_vals = []

    for l in lines:

        x1,y1,x2,y2 = l[0]

        angle = abs(
            np.degrees(
                np.arctan2(y2-y1, x2-x1)
            )
        )

        length = np.hypot(
            x2-x1,
            y2-y1
        )

        if angle < 8 and length > 120:

            if abs(x1-x2) < 200:
                continue

            y_vals.append((y1+y2)//2)

            x_vals += [x1,x2]

    if len(y_vals) < 5:
        return 0

    y_vals.sort()

    gaps = np.diff(y_vals)

    if len(gaps) < 4:
        return 0

    if np.std(gaps) > 20:
        return 0

    coverage = (
        max(x_vals)-min(x_vals)
    ) / output.shape[1]

    if coverage < 0.60:
        return 0

    cv2.rectangle(
        output,
        (min(x_vals), min(y_vals)),
        (max(x_vals), max(y_vals)),
        (0,165,255),
        2
    )

    cv2.putText(
        output,
        "Faded Crossing",
        (20,80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0,165,255),
        2
    )

    return 1

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.resize(
        frame,
        (640,480)
    )

    gray, enhanced = enhance(frame)

    mask = road_mask(frame)

    output = cv2.cvtColor(
        enhanced,
        cv2.COLOR_GRAY2BGR
    )

    f = detect_faded_crossing(
        frame,
        output,
        mask
    )

    a = detect_alligator(
        gray,
        output,
        mask
    )

    e = detect_edge_cracks(
        gray,
        output,
        mask
    )

    p = detect_pothole(
        gray,
        output,
        mask
    )

    cv2.putText(
        output,
        f"A:{a} E:{e} P:{p} F:{f}",
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255,255,255),
        2
    )

    out.write(output)

    combined = np.hstack([
        frame,
        cv2.cvtColor(
            enhanced,
            cv2.COLOR_GRAY2BGR
        ),
        output
    ])

    cv2.imshow(
        "Input | Enhanced | Output",
        cv2.resize(combined, (900,300))
    )

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
out.release()

cv2.destroyAllWindows()

print("FULL PIPELINE EXECUTION COMPLETED")

FULL PIPELINE EXECUTION COMPLETED
